<a href="https://colab.research.google.com/github/dhwlxor/My-Ropo/blob/main/Automated_Siemens_Star_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q svglib reportlab ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 94.3 MB/s eta 0:00:00


In [ ]:
import io
import os
import gc
from math import pi, sin, cos, sqrt
from IPython.display import SVG, display, clear_output
import ipywidgets as widgets
from reportlab.graphics import renderPDF, renderPM
from svglib.svglib import svg2rlg

# 구글 코랩 다운로드 모듈 불러오기
try:
    from google.colab import files
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

PAGE_WIDTH = 900

RGB_TABLE = [
    [115, 82, 68], [194, 150, 130], [ 98, 122, 157], [ 87, 108, 67],
    [133, 128, 177], [103, 189, 170], [214, 126, 44], [ 80,  91, 166],
    [193,  90, 99], [ 94,  60, 108], [157, 188, 64], [224, 163, 46],
    [ 56,  61, 150], [ 70, 148, 73], [175,  54, 60], [231, 199, 31],
    [187,  86, 149], [  8, 133, 161], [243, 243, 242], [200, 200, 200],
    [160, 160, 160], [122, 122, 121], [ 85,  85, 85], [ 52,  52, 52],
]

SVG_COLORS = ["black", "white", "red", "blue", "green", "cyan", "yellow", "magenta"]

O = chr(60)  # '<'
C = chr(62)  # '>'

def star(xc, yc, radius, cycles=72, colors=['black', 'white']):
    phi = pi / cycles / len(colors)
    polygons = []
    for n in range(cycles):
        for k, col in enumerate(colors):
            angle = 360.0 * (n + k / len(colors)) / cycles
            x1 = xc - radius * sin(phi)
            y1 = yc + radius * cos(phi)
            x2 = xc + radius * sin(phi)
            y2 = yc + radius * cos(phi)
            polygons.append(
                f'{O}polygon points="{xc:.3f},{yc:.3f} {x1:.3f},{y1:.3f} {x2:.3f},{y2:.3f}" '
                f'style="fill:{col}" transform="rotate({angle:.3f}, {xc:.3f}, {yc:.3f})"/{C}'
            )
    return "\n".join(polygons)

def lphcircles(xc, yc, radius, cycles, colors, bb_height):
    lph = [10, 20, 50, 100, 200, 500, 1000, 2000, 5000]
    r_list = [bb_height * cycles * len(colors) / (2 * pi * val) for val in lph]
    circles = [
        f'{O}circle cx="{xc:.3f}" cy="{yc:.3f}" r="{r_val:.3f}" fill="none" stroke="gray" stroke-width="1"/{C}'
        for r_val in r_list if r_val > 10 and radius > r_val
    ]
    return "\n".join(circles)

def colorchecker(bw, bh):
    rects = []
    for col in range(4):
        for row in range(6):
            r, g, b = RGB_TABLE[row + 6 * col]
            x = bw * ((0.5 + col) + 15 * (col > 1))
            y = bh * (7 + row)
            rects.append(
                f'{O}rect width="{bw:.3f}" height="{bh:.3f}" x="{x:.3f}" y="{y:.3f}" '
                f'style="fill:rgb({r},{g},{b});stroke-width:2;stroke:black"/{C}'
            )
    return "\n".join(rects)

def SiemensStar(cycles=144, aspect=4/3, colors=['black', 'white'], colorcheck=True, cstars=True, lph=True):
    bb_width = PAGE_WIDTH
    bb_height = bb_width / aspect
    r = bb_height * min(0.5, (sqrt(1 + aspect**2)) / 4)

    svg_parts = []
    svg_parts.append(f'{O}svg xmlns="http://www.w3.org/2000/svg" width="{bb_width:.3f}" height="{bb_height:.3f}" style="border:solid 1px"{C}')
    svg_parts.append(star(bb_width / 2, bb_height / 2, r, cycles, colors))

    if lph:
        res = lphcircles(bb_width / 2, bb_height / 2, r, cycles, colors, bb_height)
        if res: svg_parts.append(res)

    if cstars:
        for x in (0, bb_width):
            for y in (0, bb_height):
                svg_parts.append(star(x, y, r, cycles, colors))
                if lph:
                    res = lphcircles(x, y, r, cycles, colors, bb_height)
                    if res: svg_parts.append(res)

    if colorcheck:
        svg_parts.append(colorchecker(0.05 * bb_width, 0.05 * bb_height))

    svg_parts.append(f'{O}/svg{C}')
    return "\n".join(p for p in svg_parts if p)

# --- UI 컨트롤 위젯 ---
w_cycles = widgets.IntSlider(min=12, max=600, value=144, step=2, description="Color Cycles")
w_aspect = widgets.Dropdown(options=["16/9", "3/2", "4/3", "1/1"], value="4/3", description="Aspect Ratio")
w_color1 = widgets.Dropdown(options=SVG_COLORS, value="black", description="Color A")
w_color2 = widgets.Dropdown(options=SVG_COLORS, value="white", description="Color B")
w_colorcheck = widgets.Checkbox(value=True, description="Color Checker")
w_cstars = widgets.Checkbox(value=True, description="Corner Stars")
w_clines = widgets.Checkbox(value=True, description="LPH Lines")

w_fmt_svg = widgets.Checkbox(value=True, description="SVG")
w_fmt_png = widgets.Checkbox(value=True, description="PNG")
w_fmt_pdf = widgets.Checkbox(value=True, description="PDF")

btn_generate = widgets.Button(description="차트 생성 (미리보기)", button_style="info", icon="eye")
btn_save = widgets.Button(description="파일 저장 및 내 PC 다운로드", button_style="success", icon="download")
w_out = widgets.Output()

current_svg_code = None

def get_next_file_number():
    num = 1
    while True:
        exists = any(os.path.exists(f"star_{num}.{ext}") for ext in ["svg", "png", "pdf"])
        if not exists:
            return num
        num += 1

def on_click_generate(b):
    global current_svg_code
    with w_out:
        clear_output(wait=True)
        current_svg_code = SiemensStar(
            cycles=w_cycles.value,
            aspect=eval(w_aspect.value),
            colors=[w_color1.value, w_color2.value],
            colorcheck=w_colorcheck.value,
            cstars=w_cstars.value,
            lph=w_clines.value,
        )
        display(SVG(current_svg_code))
        print("👁️ 차트 미리보기가 표시되었습니다. 오른쪽 버튼을 눌러 내 컴퓨터로 내려받으세요.")

def on_click_save(b):
    global current_svg_code
    with w_out:
        if current_svg_code is None:
            current_svg_code = SiemensStar(
                cycles=w_cycles.value,
                aspect=eval(w_aspect.value),
                colors=[w_color1.value, w_color2.value],
                colorcheck=w_colorcheck.value,
                cstars=w_cstars.value,
                lph=w_clines.value,
            )
            clear_output(wait=True)
            display(SVG(current_svg_code))

        num = get_next_file_number()
        saved_files = []

        # 1. SVG 저장
        if w_fmt_svg.value:
            filename = f"star_{num}.svg"
            with open(filename, "w", encoding="utf-8") as f:
                f.write(current_svg_code)
            saved_files.append(filename)

        # 2. PNG / PDF 저장
        if w_fmt_png.value or w_fmt_pdf.value:
            try:
                svg_bytes = io.BytesIO(current_svg_code.encode("utf-8"))
                chart = svg2rlg(svg_bytes)
                if chart is not None:
                    if w_fmt_pdf.value:
                        filename = f"star_{num}.pdf"
                        renderPDF.drawToFile(chart, filename)
                        saved_files.append(filename)
                    if w_fmt_png.value:
                        filename = f"star_{num}.png"
                        img = renderPM.drawToPIL(chart)
                        img.save(filename, "PNG")
                        saved_files.append(filename)
                svg_bytes.close()
                del chart
                gc.collect()
            except Exception as e:
                print(f"⚠️ 변환 저장 중 오류 발생: {e}")

        # 3. 내 컴퓨터로 즉시 자동 다운로드 처리
        if saved_files:
            print(f"💾 서버 저장 완료: {', '.join(saved_files)}")
            for f_name in saved_files:
                abs_path = os.path.abspath(f_name)
                print(f"📍 서버 전체 경로: {abs_path}")
                if IS_COLAB:
                    try:
                        print(f"⬇️ 내 컴퓨터로 다운로드 중... ({f_name})")
                        files.download(f_name)
                    except Exception:
                        pass
        else:
            print("⚠️ 저장할 파일 형식을 하나 이상 선택해주세요.")

btn_generate.on_click(on_click_generate)
btn_save.on_click(on_click_save)

ui_layout = widgets.VBox([
    w_cycles, w_aspect, w_color1, w_color2,
    widgets.HBox([w_colorcheck, w_cstars, w_clines]),
    widgets.HBox([widgets.Label("저장 형식 선택:"), w_fmt_svg, w_fmt_png, w_fmt_pdf]),
    widgets.HBox([btn_generate, btn_save])
])

display(ui_layout, w_out)

Output()